# Student B – Statistical Summary

This notebook presents the statistical analysis of the cleaned Student Performance dataset.

The analysis includes:

- Summary statistics
- Normality testing
- Pearson correlation
- Spearman correlation
- Interpretation of the main statistical findings

In [3]:
from pathlib import Path
import sys
import pandas as pd
from scipy import stats

PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_processed_data

df = load_processed_data()

REPORTS_DIR = PROJECT_ROOT / "reports"

df.head()

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,4,0,11,11
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,2,9,11,11
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,6,12,13,12
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,0,14,14,14
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,0,11,13,13


## 1. Summary Statistics

The following section calculates descriptive statistics for all numeric variables, including mean, median, mode, standard deviation, variance, skewness, kurtosis, quartiles, range, IQR, coefficient of variation, and missing percentage.

In [4]:
numeric_cols = df.select_dtypes(include=["number"]).columns

summary = pd.DataFrame()

summary["Mean"] = df[numeric_cols].mean()
summary["Median"] = df[numeric_cols].median()
summary["Mode"] = df[numeric_cols].mode().iloc[0]
summary["Std"] = df[numeric_cols].std()
summary["Variance"] = df[numeric_cols].var()
summary["Skewness"] = df[numeric_cols].skew()
summary["Kurtosis"] = df[numeric_cols].kurt()

summary["Q1"] = df[numeric_cols].quantile(0.25)
summary["Q3"] = df[numeric_cols].quantile(0.75)

summary["Range"] = df[numeric_cols].max() - df[numeric_cols].min()
summary["IQR"] = summary["Q3"] - summary["Q1"]

summary["CV"] = summary["Std"] / summary["Mean"] * 100
summary["Missing %"] = df[numeric_cols].isnull().mean() * 100

summary.to_csv(REPORTS_DIR / "summary_statistics.csv")

summary

,Mean,Median,Mode,Std,Variance,Skewness,Kurtosis,Q1,Q3,Range,IQR,CV,Missing %
age,16.744222,17.0,17,1.218138,1.483859,0.416795,0.071509,16.0,18.0,7,2.0,7.274973,0.0
Medu,2.514638,2.0,2,1.134552,1.287208,-0.029950,-1.260619,2.0,4.0,4,2.0,45.117907,0.0
Fedu,2.306626,2.0,2,1.099931,1.209848,0.215343,-1.109241,1.0,3.0,4,2.0,47.685716,0.0
traveltime,1.568567,1.0,1,0.748660,0.560492,1.247648,1.108865,1.0,2.0,3,1.0,47.728919,0.0
studytime,1.930663,2.0,2,0.829510,0.688086,0.699619,0.037846,1.0,2.0,3,1.0,42.965021,0.0
failures,0.221880,0.0,0,0.593235,0.351928,3.092699,9.824409,0.0,0.0,3,0.0,267.367786,0.0
famrel,3.930663,4.0,4,0.955717,0.913395,-1.105934,1.348973,4.0,5.0,4,1.0,24.314397,0.0
freetime,3.180277,3.0,3,1.051093,1.104796,-0.181277,-0.396959,3.0,4.0,4,1.0,33.050343,0.0
goout,3.184900,3.0,3,1.175766,1.382426,-0.008580,-0.865454,2.0,4.0,4,2.0,36.916894,0.0
Dalc,1.502311,1.0,1,0.924834,0.855319,2.141913,4.349297,1.0,2.0,4,1.0,61.560774,0.0


### Interpretation of Summary Statistics

The average final grade (G3) is approximately 11.91, with a median of 12. The average first-period grade (G1) is approximately 11.40, while the average second-period grade (G2) is approximately 11.57.

Student absences have a mean of approximately 3.66 and a median of 2. The positive skewness of absences indicates that most students have relatively few absences, while a smaller number of students have many absences.

The failures variable is also strongly positively skewed. Its median and mode are both 0, showing that most students have no previous class failures.

There are no missing values in the numeric variables.

## 2. Normality Testing

The Shapiro-Wilk test is used to examine whether selected quantitative variables follow a normal distribution.

The null hypothesis states that the data are normally distributed. A p-value below 0.05 indicates a statistically significant departure from normality.

In [5]:
normality_cols = ["age", "absences", "G1", "G2", "G3"]

results = {
    "Column": [],
    "Statistic": [],
    "P-value": [],
    "Normal?": []
}

for col in normality_cols:
    data = df[col].dropna()
    stat, p_value = stats.shapiro(data)

    results["Column"].append(col)
    results["Statistic"].append(stat)
    results["P-value"].append(p_value)
    results["Normal?"].append(p_value > 0.05)

normality_df = pd.DataFrame(results)

normality_df.to_csv(
    REPORTS_DIR / "normality_tests.csv",
    index=False
)

normality_df

,Column,Statistic,P-value,Normal?
0,age,0.915595,1.519202e-18,False
1,absences,0.771744,4.522443e-29,False
2,G1,0.985539,4.933521e-06,False
3,G2,0.961667,5.583292e-12,False
4,G3,0.925981,2.415986e-17,False


### Interpretation of Normality Tests

The Shapiro-Wilk tests for age, absences, G1, G2, and G3 produced p-values below 0.05. Therefore, these variables show statistically significant departures from a normal distribution.

Ordinal variables such as Medu, Fedu, studytime, health, Dalc, and Walc were not included in the normality test because they represent ordered categories rather than continuous measurements.

## 3. Correlation Analysis

Both Pearson and Spearman correlations are calculated.

Pearson correlation measures linear relationships between numeric variables, while Spearman correlation is rank-based and is especially useful for ordinal variables or variables that do not satisfy normality assumptions.

In [6]:
pearson = df[numeric_cols].corr()
spearman = df[numeric_cols].corr(method="spearman")

pearson.to_csv(REPORTS_DIR / "pearson_correlation.csv")
spearman.to_csv(REPORTS_DIR / "spearman_correlation.csv")

pearson

,age,Medu,Fedu,traveltime,studytime,failures,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
age,1.000000,-0.107832,-0.121050,0.034490,-0.008415,0.319968,-0.020559,-0.004910,0.112805,0.134768,0.086357,-0.008750,0.149998,-0.174322,-0.107119,-0.106505
Medu,-0.107832,1.000000,0.647477,-0.265079,0.097006,-0.172210,0.024421,-0.019686,0.009536,-0.007018,-0.019766,0.004614,-0.008577,0.260472,0.264035,0.240151
Fedu,-0.121050,0.647477,1.000000,-0.208288,0.050400,-0.165915,0.020256,0.006841,0.027690,0.000061,0.038445,0.044910,0.029859,0.217501,0.225139,0.211800
traveltime,0.034490,-0.265079,-0.208288,1.000000,-0.063154,0.097730,-0.009521,0.000937,0.057454,0.092824,0.057007,-0.048261,-0.008149,-0.154120,-0.154489,-0.127173
studytime,-0.008415,0.097006,0.050400,-0.063154,1.000000,-0.147441,-0.004127,-0.068829,-0.075442,-0.137585,-0.214925,-0.056433,-0.118389,0.260875,0.240498,0.249789
failures,0.319968,-0.172210,-0.165915,0.097730,-0.147441,1.000000,-0.062645,0.108995,0.045078,0.105949,0.082266,0.035588,0.122779,-0.384210,-0.385782,-0.393316
famrel,-0.020559,0.024421,0.020256,-0.009521,-0.004127,-0.062645,1.000000,0.129216,0.089707,-0.075767,-0.093511,0.109559,-0.089534,0.048795,0.089588,0.063361
freetime,-0.004910,-0.019686,0.006841,0.000937,-0.068829,0.108995,0.129216,1.000000,0.346352,0.109904,0.120244,0.084526,-0.018716,-0.094497,-0.106678,-0.122705
goout,0.112805,0.009536,0.027690,0.057454,-0.075442,0.045078,0.089707,0.346352,1.000000,0.245126,0.388680,-0.015741,0.085374,-0.074053,-0.079469,-0.087641
Dalc,0.134768,-0.007018,0.000061,0.092824,-0.137585,0.105949,-0.075767,0.109904,0.245126,1.000000,0.616561,0.059067,0.172952,-0.195171,-0.189480,-0.204719


### Spearman Correlation Matrix

In [7]:
spearman

,age,Medu,Fedu,traveltime,studytime,failures,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
age,1.000000,-0.102288,-0.110211,0.067122,0.016959,0.290735,-0.019374,-0.009869,0.130536,0.081319,0.094343,-0.018051,0.124260,-0.167373,-0.105595,-0.066277
Medu,-0.102288,1.000000,0.647194,-0.263289,0.098415,-0.208240,0.025087,-0.027895,0.010205,0.001961,-0.018234,0.016112,-0.006011,0.276400,0.285642,0.283925
Fedu,-0.110211,0.647194,1.000000,-0.222034,0.069080,-0.161312,0.021284,-0.000151,0.028787,-0.004897,0.029726,0.046351,0.032025,0.234951,0.246285,0.234633
traveltime,0.067122,-0.263289,-0.222034,1.000000,-0.089387,0.123614,-0.025649,-0.001049,0.040714,0.068463,0.031517,-0.063844,0.022924,-0.166226,-0.166901,-0.146948
studytime,0.016959,0.098415,0.069080,-0.089387,1.000000,-0.160307,0.019370,-0.076496,-0.082318,-0.171309,-0.222088,-0.076732,-0.116945,0.271412,0.259252,0.274712
failures,0.290735,-0.208240,-0.161312,0.123614,-0.160307,1.000000,-0.058723,0.100437,0.041667,0.108862,0.064746,0.041132,0.120908,-0.432432,-0.435741,-0.448360
famrel,-0.019374,0.025087,0.021284,-0.025649,0.019370,-0.058723,1.000000,0.144123,0.087775,-0.097529,-0.102033,0.092542,-0.103905,0.026310,0.058783,0.047755
freetime,-0.009869,-0.027895,-0.000151,-0.001049,-0.076496,0.100437,0.144123,1.000000,0.354345,0.127172,0.120148,0.095105,-0.028479,-0.105119,-0.120963,-0.128375
goout,0.130536,0.010205,0.028787,0.040714,-0.082318,0.041667,0.087775,0.354345,1.000000,0.233977,0.372455,-0.012122,0.103874,-0.078216,-0.111697,-0.104967
Dalc,0.081319,0.001961,-0.004897,0.068463,-0.171309,0.108862,-0.097529,0.127172,0.233977,1.000000,0.613056,0.084946,0.104280,-0.198476,-0.200592,-0.208394


## 4. Strongest Correlations

In [8]:
pearson_pairs = []

for i in range(len(pearson.columns)):
    for j in range(i + 1, len(pearson.columns)):
        corr_value = pearson.iloc[i, j]

        pearson_pairs.append({
            "Variable 1": pearson.columns[i],
            "Variable 2": pearson.columns[j],
            "Correlation": corr_value,
            "Strength": abs(corr_value)
        })

pearson_pairs = sorted(
    pearson_pairs,
    key=lambda x: x["Strength"],
    reverse=True
)

top_pearson = pd.DataFrame(pearson_pairs[:5])
top_pearson

,Variable 1,Variable 2,Correlation,Strength
0,G2,G3,0.918548,0.918548
1,G1,G2,0.864982,0.864982
2,G1,G3,0.826387,0.826387
3,Medu,Fedu,0.647477,0.647477
4,Dalc,Walc,0.616561,0.616561


In [9]:
spearman_pairs = []

for i in range(len(spearman.columns)):
    for j in range(i + 1, len(spearman.columns)):
        corr_value = spearman.iloc[i, j]

        spearman_pairs.append({
            "Variable 1": spearman.columns[i],
            "Variable 2": spearman.columns[j],
            "Correlation": corr_value,
            "Strength": abs(corr_value)
        })

spearman_pairs = sorted(
    spearman_pairs,
    key=lambda x: x["Strength"],
    reverse=True
)

top_spearman = pd.DataFrame(spearman_pairs[:5])
top_spearman

,Variable 1,Variable 2,Correlation,Strength
0,G2,G3,0.944451,0.944451
1,G1,G2,0.893065,0.893065
2,G1,G3,0.883288,0.883288
3,Medu,Fedu,0.647194,0.647194
4,Dalc,Walc,0.613056,0.613056


## 5. Key Statistical Findings

### Grade Relationships

The strongest Pearson correlation is between G2 and G3 (r ≈ 0.919). This indicates that students with higher second-period grades tend to also have higher final grades.

G1 and G2 also show a strong positive correlation (r ≈ 0.865), while G1 and G3 have a strong positive correlation (r ≈ 0.826). Therefore, student grades across the three grading periods are strongly associated.

### Parents' Education

Medu and Fedu show a positive relationship. Because these variables represent ordinal education levels, the Spearman correlation is particularly useful. Their Spearman correlation is approximately 0.647, indicating that higher levels of mother's education tend to be associated with higher levels of father's education.

### Alcohol Consumption

Dalc and Walc also show a positive Spearman correlation of approximately 0.613. This suggests that students who report higher workday alcohol consumption also tend to report higher weekend alcohol consumption.

### Normality

The Shapiro-Wilk tests for age, absences, G1, G2, and G3 resulted in p-values below 0.05, indicating departures from normality. Therefore, Spearman correlation provides a useful non-parametric comparison alongside Pearson correlation.

### Conclusion

The most important finding is the strong relationship among G1, G2, and G3. Earlier academic performance is strongly associated with later academic performance. However, correlation represents association only and does not prove that one variable causes another.